Task 1 – Bubble Chart Analysis

## Objective

To visualize the relationship between App Size and Average Rating using a Bubble Chart, where bubble size represents the number of installs.

## Dataset

Google Play Store Dataset

## Transformations Applied

* Cleaned Install counts and converted to numeric format
* Converted App Size to MB
* Applied category filtering and translation requirements
* Scaled bubble sizes based on install counts

## KPI Measured

* App Rating
* App Size (MB)
* Total Installs

## Visualization

Bubble Chart with category-wise comparison and install-based bubble scaling.

## Time Restriction

The chart is available only during the specified time window as required in the task i.e. 5pm-7pm IST.


In [1]:
print("Jupyter is working!")

Jupyter is working!


In [2]:
import os
print(os.getcwd())
print(os.listdir())

C:\Users\HP
['.copilot', '.idlerc', '.ipynb_checkpoints', '.ipython', '.jupyter', '.matplotlib', '.streamlit', '.TurboVPN', '.vscode', '.vscode-shared', 'AppData', 'Application Data', 'Contacts', 'Cookies', 'Documents', 'Downloads', 'Favorites', 'Links', 'Local Settings', 'Microsoft', 'Music', 'My Documents', 'NetHood', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{b69bf1cb-1514-11f0-9673-a9d542314832}.TM.blf', 'NTUSER.DAT{b69bf1cb-1514-11f0-9673-a9d542314832}.TMContainer00000000000000000001.regtrans-ms', 'NTUSER.DAT{b69bf1cb-1514-11f0-9673-a9d542314832}.TMContainer00000000000000000002.regtrans-ms', 'ntuser.ini', 'OneDrive', 'PrintHood', 'Recent', 'Saved Games', 'Searches', 'SendTo', 'Start Menu', 'Templates', 'Untitled Folder', 'Untitled.ipynb', 'Videos']


In [4]:
import pandas as pd

apps_df = pd.read_csv(r"C:\Users\HP\Downloads\archive\googleplaystore.csv")
reviews_df = pd.read_csv(r"C:\Users\HP\Downloads\archive\googleplaystore_user_reviews.csv")

In [5]:
print("Apps Shape:", apps_df.shape)
print("Reviews Shape:", reviews_df.shape)

Apps Shape: (10841, 13)
Reviews Shape: (64295, 5)


In [6]:
print("Apps Columns:")
print(apps_df.columns.tolist())

print("\nReviews Columns:")
print(reviews_df.columns.tolist())

Apps Columns:
['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type', 'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver', 'Android Ver']

Reviews Columns:
['App', 'Translated_Review', 'Sentiment', 'Sentiment_Polarity', 'Sentiment_Subjectivity']


In [7]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [8]:
reviews_df.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000


In [9]:
apps_df.columns.tolist()
reviews_df.columns.tolist()

['App',
 'Translated_Review',
 'Sentiment',
 'Sentiment_Polarity',
 'Sentiment_Subjectivity']

In [10]:
import numpy as np

# Copy data
apps = apps_df.copy()
reviews = reviews_df.copy()

# Clean Reviews column
apps['Reviews'] = pd.to_numeric(apps['Reviews'], errors='coerce')

# Clean Rating column
apps['Rating'] = pd.to_numeric(apps['Rating'], errors='coerce')

# Clean Installs column
apps['Installs'] = (
    apps['Installs']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('+', '', regex=False)
)

apps['Installs'] = pd.to_numeric(apps['Installs'], errors='coerce')

# Clean Size column
apps['Size'] = apps['Size'].astype(str)

apps['Size_MB'] = (
    apps['Size']
    .str.replace('M', '', regex=False)
)

apps['Size_MB'] = pd.to_numeric(apps['Size_MB'], errors='coerce')

# Convert sentiment subjectivity
reviews['Sentiment_Subjectivity'] = pd.to_numeric(
    reviews['Sentiment_Subjectivity'],
    errors='coerce'
)

print("Cleaning complete")

Cleaning complete


In [11]:
apps[['App','Rating','Reviews','Installs','Size_MB']].head()

,App,Rating,Reviews,Installs,Size_MB
0,Photo Editor & Candy Camera & Grid & ScrapBook,4.1,159.0,10000.0,19.0
1,Coloring book moana,3.9,967.0,500000.0,14.0
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",4.7,87510.0,5000000.0,8.7
3,Sketch - Draw & Paint,4.5,215644.0,50000000.0,25.0
4,Pixel Draw - Number Art Coloring Book,4.3,967.0,100000.0,2.8


In [12]:
def convert_size(size):
    if pd.isna(size):
        return np.nan

    size = str(size)

    if 'M' in size:
        return float(size.replace('M', ''))

    elif 'k' in size:
        return float(size.replace('k', '')) / 1024

    else:
        return np.nan


apps['Size_MB'] = apps['Size'].apply(convert_size)

In [13]:
apps[['Size', 'Size_MB']].sample(10)

,Size,Size_MB
8644,3.0M,3.000000
5174,29M,29.000000
9058,17M,17.000000
2041,20M,20.000000
7208,5.4M,5.400000
10290,7.4M,7.400000
4918,26M,26.000000
6351,350k,0.341797
7819,14M,14.000000
6207,Varies with device,NaN


In [14]:
subjectivity_df = (
    reviews.groupby('App', as_index=False)
    ['Sentiment_Subjectivity']
    .mean()
)

In [15]:
subjectivity_df.head()

,App,Sentiment_Subjectivity
0,10 Best Foods for You,0.495455
1,104 找工作 - 找工作 找打工 找兼職 履歷健檢 履歷診療室,0.545516
2,11st,0.443957
3,1800 Contacts - Lens Store,0.591098
4,1LINE – One Line with One Touch,0.557315


In [16]:
merged_df = pd.merge(
    apps,
    subjectivity_df,
    on='App',
    how='inner'
)

In [17]:
print(merged_df.shape)

(1532, 15)


In [18]:
selected_categories = [
    'GAME',
    'BEAUTY',
    'BUSINESS',
    'COMICS',
    'COMMUNICATION',
    'DATING',
    'ENTERTAINMENT',
    'SOCIAL',
    'EVENTS'
]

In [19]:
filtered_df = merged_df[
    (merged_df['Rating'] > 3.5) &
    (merged_df['Reviews'] > 500) &
    (merged_df['Installs'] > 50000) &
    (merged_df['Sentiment_Subjectivity'] > 0.5) &
    (merged_df['Category'].isin(selected_categories)) &
    (~merged_df['App'].str.contains('S', case=False, na=False)) &
    (merged_df['Size_MB'].notna())
].copy()

In [20]:
print("Filtered Rows:", len(filtered_df))

filtered_df[['App',
             'Category',
             'Rating',
             'Reviews',
             'Installs',
             'Size_MB',
             'Sentiment_Subjectivity']].head()

Filtered Rows: 43


,App,Category,Rating,Reviews,Installs,Size_MB,Sentiment_Subjectivity
55,Google Primer,BUSINESS,4.4,62272.0,10000000.0,18.000000,0.675000
58,Call Blocker,BUSINESS,4.6,188841.0,5000000.0,3.200000,0.655431
134,Call Blocker,COMMUNICATION,4.1,17529.0,1000000.0,10.000000,0.655431
135,"CallApp: Caller ID, Blocker & Phone Call Recorder",COMMUNICATION,4.4,483565.0,10000000.0,20.000000,0.506481
140,Caller ID +,COMMUNICATION,4.0,9498.0,1000000.0,0.115234,0.600000


In [21]:
print("Filtered Rows:", len(filtered_df))

Filtered Rows: 43


In [53]:
category_translation = {
    'BEAUTY': 'सौंदर्य',            # Hindi
    'BUSINESS': 'Business (Tamil)', # Tamil
    'DATING': 'Partnersuche'        # German
}

filtered_df['Category_Display'] = (
    filtered_df['Category']
    .replace(category_translation)
)

In [54]:
filtered_df[['Category','Category_Display']].drop_duplicates()

,Category,Category_Display
55,BUSINESS,Business (Tamil)
134,COMMUNICATION,COMMUNICATION
148,DATING,Partnersuche
273,ENTERTAINMENT,ENTERTAINMENT
539,GAME,GAME
848,SOCIAL,SOCIAL


In [24]:
filtered_df['Color'] = filtered_df['Category'].apply(
    lambda x: 'pink' if x == 'GAME' else 'skyblue'
)

In [25]:
filtered_df[['Category', 'Color']].head()

,Category,Color
55,BUSINESS,skyblue
58,BUSINESS,skyblue
134,COMMUNICATION,skyblue
135,COMMUNICATION,skyblue
140,COMMUNICATION,skyblue


In [27]:
show_chart = True

In [29]:
import matplotlib as mpl

mpl.rcParams['font.family'] = ['Noto Sans', 'Noto Sans CJK JP']

In [31]:
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Arial Unicode MS'

In [33]:
plt.rcParams['font.family'] = 'Segoe UI'

In [66]:
from datetime import datetime
import pytz

ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist)

if 17 <= current_time.hour < 19:

    import matplotlib.pyplot as plt
    import warnings

    warnings.filterwarnings("ignore")

    fig, ax = plt.subplots(figsize=(18, 8))

    for category in filtered_df['Category_Display'].unique():

        temp = filtered_df[
            filtered_df['Category_Display'] == category
        ]

        sizes = (
            temp['Installs']
            / filtered_df['Installs'].max()
            * 3500
        )

        ax.scatter(
            temp['Size_MB'],
            temp['Rating'],
            s=sizes,
            c=temp['Color'],
            alpha=0.6,
            edgecolors='black',
            linewidths=0.5,
            label=category
        )

    ax.set_xlabel("App Size (MB)")
    ax.set_ylabel("Average Rating")
    ax.set_title("App Size vs Rating (Bubble Size = Number of Installs)")
    ax.grid(True, alpha=0.3)

    ax.legend(
        title="Category",
        loc="upper center",
        bbox_to_anchor=(0.5, -0.12),
        ncol=3
    )

    plt.subplots_adjust(bottom=0.22)
    plt.show()

else:
    print("Bubble chart is available only between 5 PM IST and 7 PM IST.")

Bubble chart is available only between 5 PM IST and 7 PM IST.
